# LAB 5 — LangFuse: Instrumentando o Pipeline Advanced RAG
## MBA RAG & CAG Aplicados a Direito e Segurança Pública — Aula 3

**Objetivo:** Adicionar observabilidade completa ao pipeline Advanced RAG usando LangFuse,
gerando traces com latência por módulo.

**Entregáveis:**
- Pipeline Advanced RAG totalmente instrumentado
- Trace completo visível no LangFuse Dashboard
- Screenshot ou JSON do trace com latência por módulo

**Tempo estimado:** 30 minutos  
**Acesso LangFuse:** https://cloud.langfuse.com (gratuito) ou self-hosted


## ⚙️ Passo 1 — Instalação e Configuração do LangFuse

In [ ]:
!pip install langfuse openai opensearch-py transformers torch -q
print("✅ Instalação concluída")

In [ ]:
import os
from langfuse import Langfuse
from langfuse.decorators import observe, langfuse_context

# Configurações do LangFuse
# Obtenha suas chaves em: https://cloud.langfuse.com
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY", "sk-lf-...")
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY", "pk-lf-...")
LANGFUSE_HOST       = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")

# Para LangFuse self-hosted no Docker:
# LANGFUSE_HOST = "http://localhost:3000"

os.environ["LANGFUSE_SECRET_KEY"] = LANGFUSE_SECRET_KEY
os.environ["LANGFUSE_PUBLIC_KEY"] = LANGFUSE_PUBLIC_KEY
os.environ["LANGFUSE_HOST"]       = LANGFUSE_HOST

try:
    langfuse = Langfuse(
        secret_key=LANGFUSE_SECRET_KEY,
        public_key=LANGFUSE_PUBLIC_KEY,
        host=LANGFUSE_HOST
    )
    langfuse.auth_check()
    LANGFUSE_OK = True
    print("✅ LangFuse conectado!")
    print(f"   Dashboard: {LANGFUSE_HOST}")
except Exception as e:
    print(f"⚠️  LangFuse não disponível ({e})")
    print("   O lab executará sem envio real de traces")
    print("   Configure LANGFUSE_SECRET_KEY e LANGFUSE_PUBLIC_KEY no .env")
    LANGFUSE_OK = False

## 🔌 Passo 2 — Conectar LLM e Retriever

In [ ]:
import os, json, time
from openai import OpenAI
from typing import List, Dict
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY     = os.getenv("GROQ_API_KEY", "")
GROQ_LLM_MODEL   = os.getenv("GROQ_LLM_MODEL", "llama-3.1-8b-instant")
GROQ_BASE_URL    = "https://api.groq.com/openai/v1"
OLLAMA_BASE_URL  = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_LLM_MODEL = os.getenv("OLLAMA_LLM_MODEL", "llama3.2:3b")

llm_client = None
MODEL_NAME = None
LLM_PROVIDER = None

if GROQ_API_KEY:
    try:
        c = OpenAI(base_url=GROQ_BASE_URL, api_key=GROQ_API_KEY)
        c.chat.completions.create(model=GROQ_LLM_MODEL,
                                  messages=[{"role":"user","content":"ok"}],
                                  max_tokens=2, temperature=0.0)
        llm_client, MODEL_NAME, LLM_PROVIDER = c, GROQ_LLM_MODEL, "groq"
        print(f"✅ LLM via Groq | modelo: {MODEL_NAME}")
    except Exception as e:
        print(f"⚠️  Groq falhou ({e}); usando Ollama…")

if llm_client is None:
    try:
        c = OpenAI(base_url=f"{OLLAMA_BASE_URL}/v1", api_key="ollama")
        c.chat.completions.create(model=OLLAMA_LLM_MODEL,
                                  messages=[{"role":"user","content":"ok"}],
                                  max_tokens=2, temperature=0.0)
        llm_client, MODEL_NAME, LLM_PROVIDER = c, OLLAMA_LLM_MODEL, "ollama"
        print(f"✅ LLM via Ollama (fallback) | {MODEL_NAME}")
    except Exception as e:
        print(f"❌ Nenhum LLM disponível: {e}")

# Em traces do LangFuse, propague provider e modelo como metadata:
# metadata={"llm_provider": LLM_PROVIDER, "llm_model": MODEL_NAME}

def carregar_corpus():
    try:
        with open("../datasets/corpus_juridico_aula3.json", encoding="utf-8") as f:
            data = json.load(f)
        return data["documentos"]
    except:
        return [
            {"id": "doc_001", "texto": "A prisão em flagrante é permitida...",
             "numero": "CPP Art. 302"},
            {"id": "doc_002", "texto": "O direito ao silêncio é garantia constitucional...",
             "numero": "CF Art. 5º LXIII"},
        ]

corpus = carregar_corpus()
print(f"✅ Corpus carregado: {len(corpus)} documentos")

## 🎯 Passo 3 — Instrumentar Cada Módulo com @observe

In [ ]:
# ── MÓDULO 1: Query Rewriting (instrumentado) ─────────────────────

@observe(name="query_rewriting")
def rewrite_query_instrumentado(query: str) -> str:
    """Query rewriting instrumentado — gera um trace span automático."""
    langfuse_context.update_current_observation(
        input=query,
        metadata={"tecnica": "step_back", "modelo": MODEL_NAME}
    )
    
    prompt = f"""Você é especialista em direito processual penal.
Reformule a query em terminologia técnica jurídica:

Query: {query}

Retorne APENAS a query reformulada, sem explicação."""
    
    try:
        resp = llm_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=100, temperature=0.1
        )
        result = resp.choices[0].message.content.strip()
        usage = resp.usage
        
        langfuse_context.update_current_observation(
            output=result,
            usage={"input": usage.prompt_tokens, "output": usage.completion_tokens}
            if usage else None
        )
        return result
    except:
        fallback = f"[Reformulação jurídica de: {query}]"
        langfuse_context.update_current_observation(output=fallback)
        return fallback


# ── MÓDULO 2: Retrieval (instrumentado) ───────────────────────────

@observe(name="retrieval")
def retrieval_instrumentado(query: str, top_k: int = 20) -> List[Dict]:
    """Retrieval instrumentado com metadados."""
    langfuse_context.update_current_observation(
        input={"query": query, "top_k": top_k},
        metadata={"retriever": "corpus_local", "index": "corpus_juridico"}
    )
    
    # Busca textual simples no corpus local (substitua por OpenSearch em produção)
    query_lower = query.lower()
    scored = []
    for doc in corpus:
        texto = doc.get("texto", "").lower()
        palavras = doc.get("palavras_chave", [])
        
        # Score simples baseado em overlap de termos
        query_terms = set(query_lower.split())
        doc_terms = set(texto.split()[:50])
        kw_terms = set(" ".join(palavras).lower().split())
        
        score = len(query_terms & (doc_terms | kw_terms)) / max(len(query_terms), 1)
        scored.append({"doc": doc, "score": score})
    
    scored.sort(key=lambda x: x["score"], reverse=True)
    results = [{"id": s["doc"]["id"], "texto": s["doc"]["texto"],
                "score": round(s["score"], 4),
                "fonte": s["doc"].get("numero", s["doc"]["id"])} 
               for s in scored[:top_k]]
    
    langfuse_context.update_current_observation(
        output={"num_docs": len(results), "top_score": results[0]["score"] if results else 0},
        metadata={"documentos_recuperados": [r["fonte"] for r in results[:5]]}
    )
    return results


# ── MÓDULO 3: Reranking (instrumentado) ──────────────────────────

@observe(name="reranking")
def reranking_instrumentado(query: str, docs: List[Dict], top_k: int = 5) -> List[Dict]:
    """Reranking instrumentado — registra scores antes/depois."""
    langfuse_context.update_current_observation(
        input={"query": query, "num_docs_input": len(docs)},
        metadata={"modelo_reranker": "BAAI/bge-reranker-v2-m3", "top_k": top_k}
    )
    
    # Tentativa de reranking real; fallback para scores simulados
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        tok = AutoTokenizer.from_pretrained("BAAI/bge-reranker-v2-m3")
        model = AutoModelForSequenceClassification.from_pretrained(
            "BAAI/bge-reranker-v2-m3").eval()
        
        texts = [d["texto"] for d in docs]
        pairs = [[query, t[:512]] for t in texts]
        inputs = tok(pairs, padding=True, truncation=True, max_length=512, return_tensors="pt")
        
        with torch.no_grad():
            scores = torch.sigmoid(model(**inputs).logits.squeeze(-1)).numpy().tolist()
        
        for doc, s in zip(docs, scores):
            doc["score_reranker"] = round(float(s), 4)
    except Exception:
        # Scores simulados para demonstração
        import random; random.seed(hash(query) % 1000)
        for doc in docs:
            doc["score_reranker"] = round(random.uniform(0.3, 0.97), 4)
    
    reranked = sorted(docs, key=lambda x: x["score_reranker"], reverse=True)[:top_k]
    
    langfuse_context.update_current_observation(
        output={"num_docs_output": len(reranked),
                "top_score_reranker": reranked[0]["score_reranker"] if reranked else 0},
        metadata={"top_docs": [r["fonte"] for r in reranked]}
    )
    return reranked


# ── MÓDULO 4: Geração (instrumentado) ────────────────────────────

@observe(name="geracao_resposta")
def geracao_instrumentada(query: str, docs: List[Dict]) -> str:
    """Geração de resposta instrumentada."""
    context = "\n\n".join([
        f"[{i+1}] {d['fonte']} (score: {d['score_reranker']:.3f}):\n{d['texto'][:300]}..."
        for i, d in enumerate(docs)
    ])
    
    prompt = f"""Você é assistente jurídico especializado em direito penal brasileiro.
Baseie-se EXCLUSIVAMENTE no contexto fornecido.

CONTEXTO:
{context}

PERGUNTA: {query}

RESPOSTA:"""
    
    langfuse_context.update_current_observation(
        input={"query": query, "context_length": len(context), "num_docs": len(docs)}
    )
    
    try:
        resp = llm_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=400, temperature=0.1
        )
        result = resp.choices[0].message.content.strip()
        usage = resp.usage
        langfuse_context.update_current_observation(
            output=result,
            usage={"input": usage.prompt_tokens, "output": usage.completion_tokens}
            if usage else None
        )
        return result
    except Exception as e:
        fallback = f"[Resposta não disponível: {e}]"
        langfuse_context.update_current_observation(output=fallback)
        return fallback

print("✅ Todos os módulos instrumentados com @observe!")

## 🏃 Passo 4 — Pipeline Completo Instrumentado

In [ ]:
@observe(name="advanced_rag_pipeline")
def pipeline_advanced_rag_completo(query: str) -> Dict:
    """
    Pipeline Advanced RAG completo com observabilidade LangFuse.
    
    Cada @observe cria automaticamente um SPAN no trace:
    - trace: advanced_rag_pipeline (root)
      ├── span: query_rewriting
      ├── span: retrieval
      ├── span: reranking
      └── span: geracao_resposta
    """
    # Adicionar metadados ao trace raiz
    langfuse_context.update_current_trace(
        name=f"advanced_rag_{query[:30]}",
        session_id="aula3_lab5",
        user_id="aluno_mba",
        tags=["advanced_rag", "juridico", "aula3"],
        metadata={
            "domain": "direito_penal",
            "pipeline_version": "3.0",
            "retriever": "corpus_local",
            "reranker": "bge-reranker-v2-m3"
        }
    )
    
    t_total = time.time()
    
    # ETAPA 1: Query Rewriting
    query_reescrita = rewrite_query_instrumentado(query)
    
    # ETAPA 2: Retrieval
    docs_recuperados = retrieval_instrumentado(query_reescrita, top_k=20)
    
    # ETAPA 3: Reranking
    docs_finais = reranking_instrumentado(query, docs_recuperados, top_k=5)
    
    # ETAPA 4: Geração
    resposta = geracao_instrumentada(query, docs_finais)
    
    total_ms = (time.time() - t_total) * 1000
    
    langfuse_context.update_current_trace(
        output={"resposta": resposta, "latencia_total_ms": round(total_ms, 1)}
    )
    
    return {
        "query_original": query,
        "query_reescrita": query_reescrita,
        "num_docs_retrieved": len(docs_recuperados),
        "docs_finais": docs_finais,
        "resposta": resposta,
        "latencia_total_ms": round(total_ms, 1)
    }

print("✅ Pipeline instrumentado definido!")
print()
print("Estrutura do trace LangFuse:")
print("  📊 TRACE: advanced_rag_pipeline")
print("  ├── 🔵 SPAN: query_rewriting")
print("  ├── 🟢 SPAN: retrieval")
print("  ├── 🟡 SPAN: reranking")
print("  └── 🔴 GENERATION: geracao_resposta")

## 🚀 Passo 5 — Executar e Visualizar Traces

In [ ]:
# Executar o pipeline para 3 queries
queries_teste = [
    "O suspeito pode ficar calado no interrogatório?",
    "Qual é o prazo para audiência de custódia?",
    "O policial precisa de mandado para entrar na casa do suspeito?"
]

traces_gerados = []

print("⏳ Executando pipeline instrumentado...")
print()

for query in queries_teste:
    print(f"🔍 Query: '{query}'")
    
    resultado = pipeline_advanced_rag_completo(query)
    
    # Flush para garantir envio ao LangFuse
    if LANGFUSE_OK:
        langfuse.flush()
    
    traces_gerados.append(resultado)
    
    print(f"   Query reescrita: {resultado['query_reescrita'][:80]}...")
    print(f"   Docs recuperados: {resultado['num_docs_retrieved']}")
    print(f"   Latência total: {resultado['latencia_total_ms']}ms")
    print(f"   Resposta: {resultado['resposta'][:100]}...")
    print()

print("✅ Traces gerados!")
if LANGFUSE_OK:
    print(f"   Visualize em: {LANGFUSE_HOST}")
    print("   Navegue até 'Traces' no menu lateral")

## 📊 Passo 6 — Análise Local de Latência

In [ ]:
print("📊 ANÁLISE DE LATÊNCIA POR EXECUÇÃO")
print("=" * 55)
print()

for i, r in enumerate(traces_gerados, 1):
    print(f"Execução {i}: '{r['query_original'][:50]}...'")
    print(f"   Latência total: {r['latencia_total_ms']:.0f}ms")
    print(f"   Docs no contexto final: {len(r['docs_finais'])}")
    if r['docs_finais']:
        top_score = r['docs_finais'][0].get('score_reranker', 0)
        print(f"   Melhor score reranker: {top_score:.3f}")
    print()

latencias = [r["latencia_total_ms"] for r in traces_gerados]
print(f"📌 Estatísticas de latência:")
print(f"   Mínima:  {min(latencias):.0f}ms")
print(f"   Máxima:  {max(latencias):.0f}ms")
print(f"   Média:   {sum(latencias)/len(latencias):.0f}ms")
print()
print("✅ LAB 5 CONCLUÍDO!")
print()
print("📋 CHECKLIST DE ENTREGA:")
print("   [ ] LangFuse configurado (ou modo simulado documentado)")
print("   [ ] Pipeline com @observe em todos os módulos")
print("   [ ] 3 traces executados")
print("   [ ] Screenshot do dashboard LangFuse (ou saída do terminal)")
print("   [ ] Análise de latência por módulo registrada")